# JMH Notebook Report

Сгенерировано: **2026-03-10 17:21:55**

Источник JSON: `hw01/target/jmh-all.json`

## Метаданные запуска
- **jmhVersion**: 1.37
- **jdkVersion**: 17.0.12
- **vmName**: Java HotSpot(TM) 64-Bit Server VM
- **vmVersion**: 17.0.12+8-LTS-286
- **jvm**: /Library/Java/JavaVirtualMachines/jdk-17.0.12.jdk/Contents/Home/bin/java
- **threads**: 1
- **forks**: 1

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

json_path_candidates = [
    Path("jmh-all.json"),
    Path("/Users/makuninaarina/IdeaProjects/Databases Algo/hw01/target/jmh-all.json"),
]
json_path = next((p for p in json_path_candidates if p.exists()), None)
if json_path is None:
    raise FileNotFoundError(
        "JMH JSON not found. Checked: " + ", ".join(str(p) for p in json_path_candidates)
    )

raw = json.loads(json_path.read_text(encoding="utf-8"))
param_cols = ["baseDocs", "bucketCapacity", "docs", "n", "seed", "threshold", "wordsPerDoc"]
X_PARAM_PRIORITY = ["bucketCapacity", "n", "docs", "baseDocs", "threshold", "wordsPerDoc", "seed"]

def _parse_param_value(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return v
    if isinstance(v, str):
        s = v.strip()
        if s == "":
            return None
        try:
            return int(s)
        except ValueError:
            try:
                return float(s)
            except ValueError:
                return s
    return v

def _flatten_raw_data(raw_data):
    out = []
    def walk(x):
        if isinstance(x, (int, float)):
            out.append(float(x))
        elif isinstance(x, list):
            for item in x:
                walk(item)
    walk(raw_data)
    return out

def _try_convert_numeric_column(df, col):
    non_na = df[col].notna().sum()
    if non_na == 0:
        return
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().sum() >= non_na:
        df[col] = converted

rows = []
param_keys = set()
for r in raw:
    params = r.get("params") or {}
    pm = r.get("primaryMetric") or {}

    score_conf = pm.get("scoreConfidence") or [None, None]
    if isinstance(score_conf, list) and len(score_conf) >= 2:
        ci_low, ci_high = score_conf[0], score_conf[1]
    else:
        ci_low, ci_high = None, None

    raw_flat = _flatten_raw_data(pm.get("rawData"))
    row = {
        "benchmark": r.get("benchmark", ""),
        "bench": (r.get("benchmark", "").split(".")[-1] if r.get("benchmark") else ""),
        "mode": r.get("mode", ""),
        "score": pm.get("score"),
        "scoreError": pm.get("scoreError"),
        "ciLow": ci_low,
        "ciHigh": ci_high,
        "unit": pm.get("scoreUnit"),
        "rawCount": len(raw_flat),
        "rawMin": min(raw_flat) if raw_flat else None,
        "rawMax": max(raw_flat) if raw_flat else None,
    }

    for k, v in params.items():
        param_keys.add(k)
        row[k] = _parse_param_value(v)

    rows.append(row)

df = pd.DataFrame(rows)
param_cols = sorted(set(param_cols) | param_keys)
for p in param_cols:
    if p not in df.columns:
        df[p] = pd.NA

for col in ("score", "scoreError", "ciLow", "ciHigh", "rawMin", "rawMax"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

for p in param_cols:
    _try_convert_numeric_column(df, p)

def _should_log_scale(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return False
    mn, mx = float(s.min()), float(s.max())
    return mn > 0 and (mx / mn) >= 50.0

def _varying_params(g):
    out = []
    for p in param_cols:
        if p not in g.columns:
            continue
        s = g[p].dropna()
        if s.empty:
            continue
        if s.astype(str).nunique() > 1:
            out.append(p)
    return out

def _pick_axes(varying):
    if not varying:
        return None, None
    x_param = next((p for p in X_PARAM_PRIORITY if p in varying), varying[0])
    rest = [p for p in varying if p != x_param]
    if not rest:
        return x_param, None
    series_param = next((p for p in X_PARAM_PRIORITY if p in rest), rest[0])
    return x_param, series_param

def _aggregate(g, keys):
    if not keys:
        return pd.DataFrame([{"score": g["score"].mean(), "scoreError": g["scoreError"].mean(), "ciLow": g["ciLow"].mean(), "ciHigh": g["ciHigh"].mean()}])
    return g.groupby(keys, dropna=False, as_index=False).agg(
        score=("score", "mean"),
        scoreError=("scoreError", "mean"),
        ciLow=("ciLow", "mean"),
        ciHigh=("ciHigh", "mean"),
    )

def build_chart(g, title):
    varying = _varying_params(g)
    x_param, series_param = _pick_axes(varying)
    unit_values = g["unit"].dropna().unique()
    unit = unit_values[0] if len(unit_values) else ""
    fig = go.Figure()

    if x_param is None:
        agg = _aggregate(g, [])
        fig.add_trace(go.Bar(
            x=["all"], y=agg["score"],
            error_y=dict(type="data", array=agg["scoreError"], visible=True),
            name="score"
        ))
    elif series_param is None:
        agg = _aggregate(g, [x_param]).sort_values(x_param, na_position="last")
        fig.add_trace(go.Scatter(
            x=agg[x_param], y=agg["score"], mode="lines+markers", name="score",
            error_y=dict(type="data", array=agg["scoreError"], visible=True),
        ))
    else:
        agg = _aggregate(g, [x_param, series_param])
        for sval in sorted(agg[series_param].dropna().unique(), key=lambda v: (2, "") if v is None else ((0, float(v)) if isinstance(v, (int, float)) else (1, str(v)))):
            sub = agg[agg[series_param] == sval].sort_values(x_param, na_position="last")
            fig.add_trace(go.Scatter(
                x=sub[x_param], y=sub["score"], mode="lines+markers",
                name=f"{series_param}={sval}",
                error_y=dict(type="data", array=sub["scoreError"], visible=True),
            ))
            if sub["ciLow"].notna().all() and sub["ciHigh"].notna().all():
                x = list(sub[x_param]) + list(sub[x_param][::-1])
                y = list(sub["ciHigh"]) + list(sub["ciLow"][::-1])
                fig.add_trace(go.Scatter(
                    x=x, y=y, fill="toself", mode="lines",
                    line=dict(width=0), showlegend=False, hoverinfo="skip", opacity=0.15
                ))

    fig.update_layout(
        title=title,
        xaxis_title=x_param if x_param else "all",
        yaxis_title=f"score ({unit})" if unit else "score",
        legend_title=series_param if series_param else "Series",
        template="plotly_white",
    )
    if _should_log_scale(g["score"]):
        fig.update_yaxes(type="log")
    return fig, x_param, series_param, varying


## Сводная таблица

In [2]:
summary_cols = ['benchmark', 'mode', 'baseDocs', 'bucketCapacity', 'docs', 'n', 'seed', 'threshold', 'wordsPerDoc', 'score', 'scoreError', 'ciLow', 'ciHigh', 'unit', 'rawCount', 'rawMin', 'rawMax']
summary_df = df[summary_cols].sort_values(['benchmark', 'mode']).reset_index(drop=True)
display(summary_df)

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit,rawCount,rawMin,rawMax
0,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.006040,0.000041,0.005999,0.006081,us/op,5,0.006022,0.006049
1,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.008123,0.000277,0.007846,0.008400,us/op,5,0.008024,0.008210
2,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.006840,0.000044,0.006796,0.006884,us/op,5,0.006833,0.006860
3,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.008901,0.000140,0.008761,0.009041,us/op,5,0.008863,0.008941
4,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.008850,0.000452,0.008398,0.009303,us/op,5,0.008717,0.008970
5,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.009963,0.000190,0.009773,0.010153,us/op,5,0.009898,0.010035
6,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,162.700797,3.688991,159.011805,166.389788,ops/us,5,161.603612,163.595489
7,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,122.910936,4.160093,118.750843,127.071029,ops/us,5,121.747263,124.151320
8,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,149.419438,1.304641,148.114797,150.724079,ops/us,5,148.869749,149.737257
9,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,111.094951,3.264182,107.830769,114.359132,ops/us,5,110.318899,112.204205


## Algorithm: Extendable Hashing (ExtendableHashTableBenchmark)

### bench.ExtendableHashTableBenchmark.getHit
**mode**: `avgt`

In [3]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.getHit") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.getHit [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.006040,0.000041,0.005999,0.006081,us/op
1,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.008123,0.000277,0.007846,0.008400,us/op
2,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.006840,0.000044,0.006796,0.006884,us/op
3,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.008901,0.000140,0.008761,0.009041,us/op
4,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.008850,0.000452,0.008398,0.009303,us/op
5,bench.ExtendableHashTableBenchmark.getHit,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.009963,0.000190,0.009773,0.010153,us/op


Чтение существующего ключа остаётся очень быстрым, но постепенно дорожает при росте n и bucketCapacity. Это согласуется с реализацией.

Лучший режим для getHit в этих данных маленький бакет bucketCapacity=2, потому что чтение упирается вo внутрибакетный линейный поиск, а не в split/merge.


### bench.ExtendableHashTableBenchmark.getHit
**mode**: `thrpt`

In [4]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.getHit") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.getHit [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,162.700797,3.688991,159.011805,166.389788,ops/us
1,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,122.910936,4.160093,118.750843,127.071029,ops/us
2,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,149.419438,1.304641,148.114797,150.724079,ops/us
3,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,111.094951,3.264182,107.830769,114.359132,ops/us
4,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,8.0,NaN,1000.0,42,NaN,NaN,112.108424,1.546654,110.561770,113.655077,ops/us
5,bench.ExtendableHashTableBenchmark.getHit,thrpt,NaN,8.0,NaN,10000.0,42,NaN,NaN,94.084985,2.694119,91.390866,96.779103,ops/us


Чем медленнее чтение, тем ниже throughput. На нём особенно хорошо видно, что увеличение bucketCapacity просаживает скорость успешных чтений.
Для hit-path у расширяемого хеширования маленький бакет выгоднее большого, так как меньше слотов для линейного просмотра, выше число операций в микросекунду.

### bench.ExtendableHashTableBenchmark.getMiss
**mode**: `avgt`

In [5]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.getMiss") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.getMiss [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.004951,0.000048,0.004904,0.004999,us/op
1,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.006861,0.000442,0.006419,0.007303,us/op
2,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.006309,0.000094,0.006215,0.006402,us/op
3,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.008156,0.000436,0.007719,0.008592,us/op
4,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.009181,0.000045,0.009135,0.009226,us/op
5,bench.ExtendableHashTableBenchmark.getMiss,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.011304,0.000440,0.010864,0.011743,us/op


Miss-lookup ведёт себя похоже на hit-lookup, но в этих данных даже выглядит немного быстрее. Это не универсальное свойство miss вообще, а следствие конкретной постановки бенчмарка, так как absent-ключи заранее детерминированы как i + n, а не выбраны случайно.

Основное бутылочное горлышко то же, что и у getHit - линейный поиск внутри бакета.

### bench.ExtendableHashTableBenchmark.getMiss
**mode**: `thrpt`

In [6]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.getMiss") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.getMiss [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,200.687837,1.892563,198.795274,202.580400,ops/us
1,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,147.251007,8.765538,138.485469,156.016545,ops/us
2,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,158.077124,4.746212,153.330913,162.823336,ops/us
3,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,120.695248,9.183937,111.511311,129.879184,ops/us
4,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,8.0,NaN,1000.0,42,NaN,NaN,108.745376,1.782364,106.963013,110.527740,ops/us
5,bench.ExtendableHashTableBenchmark.getMiss,thrpt,NaN,8.0,NaN,10000.0,42,NaN,NaN,90.487264,3.481564,87.005700,93.968829,ops/us


График повторяет getMiss [avgt] в обратной шкале.
Miss-path остаётся очень быстрым, но его производительность также ухудшается на больших бакетах. То есть проблема не в проверке наличия как таковой, а в цене линейного просмотра бакета.

### bench.ExtendableHashTableBenchmark.putThenRemove
**mode**: `avgt`

In [7]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.putThenRemove") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.putThenRemove [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,34.938101,0.592831,34.345270,35.530932,us/op
1,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,265.664944,1.870342,263.794602,267.535287,us/op
2,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.562347,0.012583,0.549764,0.574930,us/op
3,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,15.501305,0.175930,15.325375,15.677235,us/op
4,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.095591,0.001085,0.094507,0.096676,us/op
5,bench.ExtendableHashTableBenchmark.putThenRemove,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,1.183300,0.016316,1.166984,1.199616,us/op


Это один из самых показательных графиков во всём отчёте. Вставка с последующим удалением резко дорожает при маленьком bucketCapacity, потому что код входит в дорогой путь splitBucket(), а после удаления ещё и пытается выполнять tryMerge(). Внутри split происходит перераздача записей, обновление директории и работа с mmap-файлом. 

### bench.ExtendableHashTableBenchmark.putThenRemove
**mode**: `thrpt`

In [8]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.putThenRemove") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.putThenRemove [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.028663,0.000167,0.028495,0.028830,ops/us
1,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.003768,0.000021,0.003747,0.003789,ops/us
2,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,1.770629,0.025239,1.745390,1.795868,ops/us
3,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.064533,0.000968,0.063565,0.065501,ops/us
4,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,8.0,NaN,1000.0,42,NaN,NaN,10.317481,0.068160,10.249321,10.385641,ops/us
5,bench.ExtendableHashTableBenchmark.putThenRemove,thrpt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.844911,0.004286,0.840625,0.849197,ops/us


Малый бакет даёт отличный поиск, но почти убивает производительность у сценария с вставкой и удалением.

Для write-heavy сценария bucketCapacity=2 - плохой выбор. На этом пути throughput определяется частотой split/merge, а не простой стоимостью put/remove.


### bench.ExtendableHashTableBenchmark.removeThenPut
**mode**: `avgt`

In [9]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.removeThenPut") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.removeThenPut [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,16.444298,0.213679,16.230619,16.657977,us/op
1,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,257.862243,5.824020,252.038224,263.686263,us/op
2,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.659877,0.010014,0.649864,0.669891,us/op
3,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,16.840745,0.512117,16.328627,17.352862,us/op
4,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.187478,0.002940,0.184538,0.190418,us/op
5,bench.ExtendableHashTableBenchmark.removeThenPut,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,1.086661,0.007134,1.079527,1.093795,us/op


Когда сначала идёт удаление, часть вставок действительно получает свободный слот, однако при больших n и маленьком бакете структурные перестройки всё равно доминируют.

Проблема сидит не в конкретном порядке двух операций, а в том, что write-path чувствителен к плотности бакетов и слишком рано платит за балансировку структуры.

### bench.ExtendableHashTableBenchmark.removeThenPut
**mode**: `thrpt`

In [10]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.removeThenPut") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.removeThenPut [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.060828,0.000844,0.059983,0.061672,ops/us
1,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.003888,0.000023,0.003865,0.003911,ops/us
2,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,1.515185,0.024110,1.491074,1.539295,ops/us
3,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.058768,0.006507,0.052261,0.065274,ops/us
4,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,8.0,NaN,1000.0,42,NaN,NaN,5.401953,0.067381,5.334572,5.469334,ops/us
5,bench.ExtendableHashTableBenchmark.removeThenPut,thrpt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.915883,0.013257,0.902626,0.929139,ops/us


Производительность рушится из-за общего поведения структуры под чередованием remove/put.

При write-heavy нагрузке производительность определяется тем, насколько часто таблица вынуждена перестраивать бакеты и директорию. Большие бакеты в этом сценарии явно устойчивее.

### bench.ExtendableHashTableBenchmark.updateExisting
**mode**: `avgt`

In [11]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.updateExisting") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.updateExisting [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,2.0,NaN,1000.0,42,NaN,NaN,0.013408,0.000201,0.013206,0.013609,us/op
1,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,2.0,NaN,10000.0,42,NaN,NaN,0.017169,0.000390,0.016780,0.017559,us/op
2,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,4.0,NaN,1000.0,42,NaN,NaN,0.016156,0.001625,0.014532,0.017781,us/op
3,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,4.0,NaN,10000.0,42,NaN,NaN,0.018334,0.000544,0.017789,0.018878,us/op
4,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,8.0,NaN,1000.0,42,NaN,NaN,0.017472,0.001154,0.016318,0.018626,us/op
5,bench.ExtendableHashTableBenchmark.updateExisting,avgt,NaN,8.0,NaN,10000.0,42,NaN,NaN,0.019509,0.000460,0.019049,0.019969,us/op


При обновлении существующего ключа структура не split-ится и не merge-ится, код просто находит ключ в бакете и переписывает value. Поэтому операция остаётся быстрой и растёт очень плавно.

### bench.ExtendableHashTableBenchmark.updateExisting
**mode**: `thrpt`

In [12]:
g = df[(df['benchmark'] == "bench.ExtendableHashTableBenchmark.updateExisting") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.ExtendableHashTableBenchmark.updateExisting [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,2.0,NaN,1000.0,42,NaN,NaN,74.981357,1.653092,73.328265,76.634449,ops/us
1,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,2.0,NaN,10000.0,42,NaN,NaN,58.689174,1.918095,56.771079,60.607269,ops/us
2,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,4.0,NaN,1000.0,42,NaN,NaN,61.410740,4.511428,56.899312,65.922168,ops/us
3,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,4.0,NaN,10000.0,42,NaN,NaN,54.892343,3.888283,51.004060,58.780626,ops/us
4,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,8.0,NaN,1000.0,42,NaN,NaN,57.470328,2.518912,54.951417,59.989240,ops/us
5,bench.ExtendableHashTableBenchmark.updateExisting,thrpt,NaN,8.0,NaN,10000.0,42,NaN,NaN,52.042932,4.582887,47.460045,56.625819,ops/us


Если не провоцировать split/merge, расширяемое хеширование работает стабильно. Это значит, что оптимизировать в первую очередь нужно ребалансировку.

## Algorithm: LSH (LshBenchmark)

### bench.LshBenchmark.addDocument
**mode**: `avgt`

In [13]:
g = df[(df['benchmark'] == "bench.LshBenchmark.addDocument") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.addDocument [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.addDocument,avgt,5000.0,NaN,NaN,NaN,42,NaN,12.0,0.022882,0.003943,0.018939,0.026824,ms/op


Добавление документа не выглядит медленным само по себе, но внутри него уже сидит тяжёлая константа из-за большого количества аллокаций и строковой обработки.

### bench.LshBenchmark.addDocument
**mode**: `thrpt`

In [14]:
g = df[(df['benchmark'] == "bench.LshBenchmark.addDocument") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.addDocument [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.addDocument,thrpt,5000.0,NaN,NaN,NaN,42,NaN,12.0,44.176749,4.477678,39.699071,48.654427,ops/ms


Вставка одного документа в индекс вполне рабочая, но здесь ещё не видно, какая часть времени уходит на текстовую подготовку, а какая — на собственно LSH-структуру.


### bench.LshBenchmark.buildIndex
**mode**: `avgt`

In [15]:
g = df[(df['benchmark'] == "bench.LshBenchmark.buildIndex") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.buildIndex [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.buildIndex,avgt,NaN,NaN,2000.0,NaN,42,NaN,12.0,39.971090,0.549092,39.421998,40.520182,ms/op
1,bench.LshBenchmark.buildIndex,avgt,NaN,NaN,10000.0,NaN,42,NaN,12.0,190.711714,1.690954,189.020760,192.402668,ms/op


Построение индекса растёт почти линейно по числу документов: 2000 документов строятся примерно за 40 ms, а 10000 примерно за 191 ms. Для такого пайплайна это хороший знак: явного квадратичного взрыва нет.  buildIndex масштабируется близко к линейному по числу документов, но с крупной константой из-за тяжёлой обработки каждого текста.

### bench.LshBenchmark.buildIndex
**mode**: `thrpt`

In [16]:
g = df[(df['benchmark'] == "bench.LshBenchmark.buildIndex") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.buildIndex [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.buildIndex,thrpt,NaN,NaN,2000.0,NaN,42,NaN,12.0,0.026818,0.000356,0.026461,0.027174,ops/ms
1,bench.LshBenchmark.buildIndex,thrpt,NaN,NaN,10000.0,NaN,42,NaN,12.0,0.005209,0.000059,0.005149,0.005268,ops/ms


Чем больше документов нужно прогнать через preprocessing и MinHash, тем меньше индексов удаётся построить за миллисекунду.
Бутылочное горлышко в цене одного документа.

### bench.LshBenchmark.candidatesQuery
**mode**: `avgt`

In [17]:
g = df[(df['benchmark'] == "bench.LshBenchmark.candidatesQuery") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.candidatesQuery [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.candidatesQuery,avgt,NaN,NaN,1000.0,NaN,42,0.9,12.0,0.019071,0.000257,0.018814,0.019328,ms/op
1,bench.LshBenchmark.candidatesQuery,avgt,NaN,NaN,3000.0,NaN,42,0.9,12.0,0.021137,0.000570,0.020567,0.021707,ms/op


Запрос кандидатов почти не меняется между docs=1000 и docs=3000. Увеличение индекса в 3 раза почти не меняет время ответа, потому что retrieval кандидатов дешёвый, а дороже всего считать подпись запроса. Стоимость запроса определяется в первую очередь построением сигнатуры запроса, а не обходом уже существующих band-таблиц. Отдельно важно, что параметр threshold в этом benchmark state есть, но в методе candidatesQuery() вообще не используется.



### bench.LshBenchmark.candidatesQuery
**mode**: `thrpt`

In [18]:
g = df[(df['benchmark'] == "bench.LshBenchmark.candidatesQuery") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.candidatesQuery [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.candidatesQuery,thrpt,NaN,NaN,1000.0,NaN,42,0.9,12.0,52.558066,0.850830,51.707235,53.408896,ops/ms
1,bench.LshBenchmark.candidatesQuery,thrpt,NaN,NaN,3000.0,NaN,42,0.9,12.0,47.158819,1.426482,45.732337,48.585302,ops/ms


Поиск кандидатов сильная сторона текущей реализации. Он хорошо масштабируется и не показывает резкого провала даже при росте объёма данных.

### bench.LshBenchmark.nearDuplicatesFullScan
**mode**: `avgt`

In [19]:
g = df[(df['benchmark'] == "bench.LshBenchmark.nearDuplicatesFullScan") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.nearDuplicatesFullScan [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.nearDuplicatesFullScan,avgt,NaN,NaN,1000.0,NaN,42,0.9,12.0,145.998555,2.458911,143.539644,148.457465,ms/op
1,bench.LshBenchmark.nearDuplicatesFullScan,avgt,NaN,NaN,3000.0,NaN,42,0.9,12.0,1440.251458,9.035529,1431.215929,1449.286987,ms/op


Очень плохо масштабируется. В коде выполняется полный двойной цикл по всем парам документов с точным Jaccard, поэтому рост корпуса быстро становится критическим. Полный перебор годится только как контрольный baseline или как проверка качества на маленьких данных. Для реального поиска near-duplicates он слишком дорогой.

### bench.LshBenchmark.nearDuplicatesFullScan
**mode**: `thrpt`

In [20]:
g = df[(df['benchmark'] == "bench.LshBenchmark.nearDuplicatesFullScan") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.nearDuplicatesFullScan [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.nearDuplicatesFullScan,thrpt,NaN,NaN,1000.0,NaN,42,0.9,12.0,0.006839,0.000038,0.006801,0.006878,ops/ms
1,bench.LshBenchmark.nearDuplicatesFullScan,thrpt,NaN,NaN,3000.0,NaN,42,0.9,12.0,0.000695,0.000011,0.000684,0.000706,ops/ms


throughput у полного перебора почти исчезает уже на 3000 документах.  Даже на небольшом росте данных он быстро становится непрактичным по пропускной способности.

### bench.LshBenchmark.nearDuplicatesLsh
**mode**: `avgt`

In [21]:
g = df[(df['benchmark'] == "bench.LshBenchmark.nearDuplicatesLsh") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.nearDuplicatesLsh [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.nearDuplicatesLsh,avgt,NaN,NaN,1000.0,NaN,42,0.9,12.0,10.591143,0.164489,10.426654,10.755632,ms/op
1,bench.LshBenchmark.nearDuplicatesLsh,avgt,NaN,NaN,3000.0,NaN,42,0.9,12.0,126.204284,10.711097,115.493187,136.915381,ms/op


LSH-поиск работает намного лучше full scan, но его рост всё равно очень тяжёлый. Причина в коде: после отбора кандидатов по полосам реализация всё равно генерирует пары внутри бакетов, устраняет дубликаты через seenPairs, считает точный Jaccard и сортирует результат. Точную проверку кандидатов тоже нужно оптимизировать.

### bench.LshBenchmark.nearDuplicatesLsh
**mode**: `thrpt`

In [22]:
g = df[(df['benchmark'] == "bench.LshBenchmark.nearDuplicatesLsh") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.LshBenchmark.nearDuplicatesLsh [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.LshBenchmark.nearDuplicatesLsh,thrpt,NaN,NaN,1000.0,NaN,42,0.9,12.0,0.091737,0.001795,0.089942,0.093532,ops/ms
1,bench.LshBenchmark.nearDuplicatesLsh,thrpt,NaN,NaN,3000.0,NaN,42,0.9,12.0,0.008535,0.000985,0.007550,0.009520,ops/ms


LSH полезен и даёт сильный выигрыш, но в текущей реализации near-duplicate search упирается в верификацию пар внутри бакетов.

## Algorithm: Perfect Hashing (PerfectHashingBenchmark)

### bench.PerfectHashingBenchmark.buildIndex
**mode**: `avgt`

In [23]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.buildIndex") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.buildIndex [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.buildIndex,avgt,NaN,NaN,NaN,10000.0,42,NaN,NaN,299.848140,18.974004,280.874137,318.822144,us/op
1,bench.PerfectHashingBenchmark.buildIndex,avgt,NaN,NaN,NaN,50000.0,42,NaN,NaN,1533.994382,25.533503,1508.460879,1559.527884,us/op


Построение идеальной хеш-таблицы закономерно дороже чтений. По коду сначала подбирается h1, пока сумма квадратов размеров бакетов не станет приемлемой, затем для каждого бакета строится вторичная таблица размера n_j^2, и для неё многократно подбирается h2, пока все ключи не разложатся без коллизий, классическая стратегия "дорого построить, дёшево читать". 

### bench.PerfectHashingBenchmark.buildIndex
**mode**: `thrpt`

In [24]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.buildIndex") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.buildIndex [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.buildIndex,thrpt,NaN,NaN,NaN,10000.0,42,NaN,NaN,0.003324,0.000075,0.003250,0.003399,ops/us
1,bench.PerfectHashingBenchmark.buildIndex,thrpt,NaN,NaN,NaN,50000.0,42,NaN,NaN,0.000649,0.000007,0.000642,0.000657,ops/us


Чем больше ключей, тем больше попыток подбора h1/h2 и тем меньше полных построений таблицы успевает выполнить benchmark.

### bench.PerfectHashingBenchmark.containsHit
**mode**: `avgt`

In [25]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.containsHit") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.containsHit [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.containsHit,avgt,NaN,NaN,NaN,10000.0,42,NaN,NaN,0.007541,0.000693,0.006849,0.008234,us/op
1,bench.PerfectHashingBenchmark.containsHit,avgt,NaN,NaN,NaN,50000.0,42,NaN,NaN,0.008374,0.000691,0.007683,0.009065,us/op


Успешная проверка наличия почти не зависит от размера таблицы. После build код делает всего два вычисления хеша и одну проверку ячейки на втором уровне, без цепочек и без линейного просмотра.

### bench.PerfectHashingBenchmark.containsHit
**mode**: `thrpt`

In [26]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.containsHit") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.containsHit [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.containsHit,thrpt,NaN,NaN,NaN,10000.0,42,NaN,NaN,132.080199,7.407340,124.672859,139.487538,ops/us
1,bench.PerfectHashingBenchmark.containsHit,thrpt,NaN,NaN,NaN,50000.0,42,NaN,NaN,122.911232,8.634065,114.277168,131.545297,ops/us


Производительность почти не проседает при увеличении числа ключей в 5 раз. Это подтверждает, что после завершения build структура работает как очень быстрый статический индекс.

### bench.PerfectHashingBenchmark.containsMiss
**mode**: `avgt`

In [27]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.containsMiss") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.containsMiss [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.containsMiss,avgt,NaN,NaN,NaN,10000.0,42,NaN,NaN,0.004616,0.000753,0.003863,0.005370,us/op
1,bench.PerfectHashingBenchmark.containsMiss,avgt,NaN,NaN,NaN,50000.0,42,NaN,NaN,0.012428,0.001308,0.011120,0.013736,us/op


Отсутствующие ключи обрабатываются менее стабильно, чем присутствующие. Разница, вероятно, идёт не от асимптотики, а от memory locality и конкретного распределения absent-key.

### bench.PerfectHashingBenchmark.containsMiss
**mode**: `thrpt`

In [28]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.containsMiss") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.containsMiss [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.containsMiss,thrpt,NaN,NaN,NaN,10000.0,42,NaN,NaN,219.168450,52.946001,166.222449,272.114451,ops/us
1,bench.PerfectHashingBenchmark.containsMiss,thrpt,NaN,NaN,NaN,50000.0,42,NaN,NaN,81.123741,8.478664,72.645077,89.602406,ops/us


Производительность miss-запросов сильно проседает на больших таблицах.

### bench.PerfectHashingBenchmark.getHit
**mode**: `avgt`

In [29]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.getHit") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.getHit [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.getHit,avgt,NaN,NaN,NaN,10000.0,42,NaN,NaN,0.008095,0.000514,0.007581,0.008610,us/op
1,bench.PerfectHashingBenchmark.getHit,avgt,NaN,NaN,NaN,50000.0,42,NaN,NaN,0.008937,0.000705,0.008232,0.009641,us/op


Успешное чтение значения ведёт себя почти так же, как containsHit, только немного дороже из-за чтения values[pos] и приведения типа.
Для чтения существующего ключа perfect hashing работает почти как O(1) на практике и масштабируется очень хорошо.

### bench.PerfectHashingBenchmark.getHit
**mode**: `thrpt`

In [30]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.getHit") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.getHit [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.getHit,thrpt,NaN,NaN,NaN,10000.0,42,NaN,NaN,121.612742,12.124912,109.487830,133.737654,ops/us
1,bench.PerfectHashingBenchmark.getHit,thrpt,NaN,NaN,NaN,50000.0,42,NaN,NaN,111.228535,12.597317,98.631218,123.825852,ops/us


После построения таблица отлично подходит для частых чтений по существующим ключам. Это один из лучших сценариев применения perfect hashing

### bench.PerfectHashingBenchmark.getMiss
**mode**: `avgt`

In [31]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.getMiss") & (df['mode'] == "avgt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.getMiss [avgt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.getMiss,avgt,NaN,NaN,NaN,10000.0,42,NaN,NaN,0.004488,0.000680,0.003808,0.005169,us/op
1,bench.PerfectHashingBenchmark.getMiss,avgt,NaN,NaN,NaN,50000.0,42,NaN,NaN,0.012307,0.001338,0.010969,0.013645,us/op


Get для отсутствующего ключа не слабое место по абсолютным цифрам, но именно здесь сильнее всего проявляется разница между маленькой и большой таблицей.


### bench.PerfectHashingBenchmark.getMiss
**mode**: `thrpt`

In [32]:
g = df[(df['benchmark'] == "bench.PerfectHashingBenchmark.getMiss") & (df['mode'] == "thrpt")].copy()
fig, x_param, series_param, varying = build_chart(g, "bench.PerfectHashingBenchmark.getMiss [thrpt]")
fig.show()
table_cols = ['benchmark', 'mode'] + [c for c in param_cols if c in g.columns] + ['score', 'scoreError', 'ciLow', 'ciHigh', 'unit']
display(g[table_cols].sort_values(table_cols[:2]).reset_index(drop=True))

,benchmark,mode,baseDocs,bucketCapacity,docs,n,seed,threshold,wordsPerDoc,score,scoreError,ciLow,ciHigh,unit
0,bench.PerfectHashingBenchmark.getMiss,thrpt,NaN,NaN,NaN,10000.0,42,NaN,NaN,222.397658,38.212051,184.185607,260.609709,ops/us
1,bench.PerfectHashingBenchmark.getMiss,thrpt,NaN,NaN,NaN,50000.0,42,NaN,NaN,81.298991,1.141566,80.157425,82.440557,ops/us


Как и в containsMiss, основной вывод здесь не про плохую структуру, а про то, что miss-path чувствителен к конкретной постановке теста и к памяти.

## Выводы

**ExtendableHashTable**

Write-path местами проваливается. Например, putThenRemove для bucketCapacity=2, n=10000 - это примерно 265.7 us/op, а при bucketCapacity=8, n=10000 уже около 1.18 us/op. То есть проблема не в самой таблице как идее, а в том, как дорого обходятся структурные перестройки.

**MinHashLshIndex**

У LSH тяжёлый nearDuplicatesLsh. Это значит, что retrieval кандидатов работает неплохо, а основная цена сидит в нормализации, генерации шинглов, расчёте сигнатуры и особенно в проверке пар внутри бакетов.

**PerfectHashTable**

Если нужно улучшить архитектурно и красиво, тогда можно оптимизировать build. Например не выделять новые массивы на каждую попытку h2, а 1 раз создать keys/used/values и на новой попытке сбрасывать только реально затронутые позиции. Можно сделать m не ровно n, а чуть больше.